# DS102 v7 — thử nghiệm, chưa xác nhận tăng điểm
Upload ds102_v7_bundle.zip. Chỉ chạy hosted Colab; không pretrained.
Nhánh Logistic Regression C=4, sqrt-balanced, blend 20%; không tìm hệ số.
So sánh v5 trên exact-group CV rồi near-duplicate-group CV; gate macro +0.003, ít nhất 2/3 fold tăng, bootstrap 95% dương, novel fake F1 không giảm quá 0.01. Dev chỉ đánh giá sau CV, không fit.
Chỉ xuất CSV candidate khi đạt gate. Không tự nộp Kaggle; điểm leaderboard chưa biết.


In [ ]:
import os, platform, zipfile, hashlib, json
from pathlib import Path
assert platform.system() == 'Linux' and os.environ.get('COLAB_RELEASE_TAG'), 'Hosted Colab only'
bundle = Path('/content/ds102_v7_bundle.zip')
if not bundle.exists():
    from google.colab import files
    uploaded = files.upload()
    assert bundle.exists(), 'Upload ds102_v7_bundle.zip'
    del uploaded
root = Path('/content/ds102_v7_bundle')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    for name in z.namelist():
        assert (root/name).resolve().is_relative_to(root.resolve()), 'Unsafe ZIP path'
    z.extractall(root)
manifest = json.loads((root/'bundle_manifest.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == expected, name
print('Bundle checked:', len(manifest), 'files. Hosted Colab CPU:', os.cpu_count())


In [ ]:
%pip -q install catboost==1.2.10 scikit-learn==1.7.2

In [ ]:
import subprocess, sys, os
env = dict(os.environ, OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', MKL_NUM_THREADS='2')
subprocess.run([sys.executable, '/content/ds102_v7_bundle/experiment_v7.py'], env=env, check=True)
